## Importação das bibliotecas

import pandas as pd

import numpy as np

In [14]:
import pandas as pd
import numpy as np

f = pd.read_csv('../data/raw/Base Varejo.csv', sep=';')

print('Arquivo inmportado com sucesso!')

Arquivo inmportado com sucesso!


# INFORMAÇÕES SOBRE A BASE DE DADOS


In [15]:
print(f'\nTotal de linhas: {f.shape[0]}\nTotal de colunas:{f.shape[1]}\n')
print('TIPOS DE DADOS')
print(f.dtypes)

print(f'\nColunas que estão totalmente vazias:\n{f.columns[f.isnull().any()].tolist()}\n')


print(f.head())


Total de linhas: 830000
Total de colunas:14

TIPOS DE DADOS
DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object

Colunas que estão totalmente vazias:
['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS   
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE   
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS   
4  01/02/2019   1000    534         M      4       1      C    175    LIMPEZA   

                

# INFORMAÇÕES SOBRE A BASE DE DADOS ANTES DA LIMPEZA

In [16]:
nulls = f.isnull().sum()
pct = nulls / len(f) * 100
duplicateds = f.duplicated().sum()
print(pd.DataFrame({'Nulos': nulls, '% Nulos': pct}))
print('\n')
print(f'Linhas potencialmente duplicadas: {duplicateds}')
print(f'Percentual de linhas: {duplicateds / len(f) * 100:.2f}%')

              Nulos  % Nulos
DATA              0      0.0
CO_ID             0      0.0
CL_ID             0      0.0
CL_GENERO         0      0.0
CL_EC             0      0.0
CL_FHL            0      0.0
CL_SEG            0      0.0
PR_ID             0      0.0
PR_CAT            0      0.0
PR_NOME           0      0.0
Unnamed: 10  830000    100.0
Unnamed: 11  830000    100.0
Unnamed: 12  830000    100.0
Unnamed: 13  830000    100.0


Linhas potencialmente duplicadas: 96553
Percentual de linhas: 11.63%


# INFORMAÇÕES SOBRE A BASE DE DADOS APÓS LIMPEZA

In [ ]:
# Limpeza da base
f = f.replace({
        'NULL': np.nan,
        'N/A': np.nan,
        '': np.nan
    })

f = f.dropna(axis=1, how='all')
duplicated_rows = f.duplicated().sum()

f = f.reset_index(drop=True)

f['PR_CAT'] = (
    f['PR_CAT']
    .fillna('#N/D')
    .str.title()
    .str.strip()
)

f['PR_NOME'] = (
    f['PR_NOME']
    .fillna('#N/D')
    .str.title()
    .str.strip()
)


In [19]:
nulls = f.isnull().sum()
pct = nulls / len(f) * 100
duplicateds = f.duplicated().sum()
print(pd.DataFrame({'Nulos': nulls, '% Nulos': pct}))
print('\n')
print(f'Linhas ptenciaçmente duplicadas: {duplicateds}')
print(f'Percentual de linhas: {duplicateds / len(f) * 100:.2f}%')

           Nulos  % Nulos
DATA           0      0.0
CO_ID          0      0.0
CL_ID          0      0.0
CL_GENERO      0      0.0
CL_EC          0      0.0
CL_FHL         0      0.0
CL_SEG         0      0.0
PR_ID          0      0.0
PR_CAT         0      0.0
PR_NOME        0      0.0


Linhas ptenciaçmente duplicadas: 96553
Percentual de linhas: 11.63%


# ANÁLISE EXPLORATÓRIA

In [ ]:
# Definição de "opções" dos valores das variáveis e renomeação das chaves do dataframe
gender = {
    'M': 'Masculino',
    'F': 'Feminino'
}

f['Gênero'] = f['CL_GENERO']
f['Gênero'] = f['CL_GENERO'].map(gender)

marital_status = {
    1: 'Casado ou União Estável',
    2: 'Divorciado',
    3: 'Separado',
    4: 'Solteiro',
    5: 'Viúvo'
}

f['Estado Civil'] = f['CL_EC'].map(marital_status)

f['Gênero'] = f['CL_GENERO'].map(gender)

f['Categoria'] = f['PR_CAT']

f['Produto'] = f['PR_NOME']

f['Classe'] = f['CL_SEG']

### 1. ANÁLISE REFERENTE A PRODUTOS INVÁLIDOS.

In [40]:
# Cópia de bases para produtos que possuam a informação #N/D
products_df = f[
(f['Categoria'] != '#N/D') &
(f['Produto'] != '#N/D')
].copy()

invalids_products = (
    (f['Categoria'] == '#N/D') |
    (f['Produto'] == '#N/D')
).sum()
print(f'\nQuantidade de produtos inválidos(\'#N/D\'): {invalids_products}')
print(f'Porcentagem de produtos inválidos(\'#N/D\'): {invalids_products / len(f) * 100:.2f}%')


Quantidade de produtos inválidos('#N/D'): 3650
Porcentagem de produtos inválidos('#N/D'): 0.44%


### 2. ANÁLISE REFERENTE A QUANTIDADE DE FILHOS POR COMPRAS.

In [39]:
number_children = f['CL_FHL']
print(f'\nQuantidade de filhos:\n\tMín:{number_children.min()} - Máx: {number_children.max()}')

average_number_children = f['CL_FHL'].mean()
print(f'\nMédia de filhos: {average_number_children:.2f}')


Quantidade de filhos:
	Mín:0 - Máx: 4

Média de filhos: 1.15


### 3. ANÁLISE REFERENTE A QUANTIDADE DE COMPRAS POR GÊNERO.

In [38]:
shopping_by_genre = f['Gênero'].value_counts().reset_index(name='Quantidade')
print(f'\nCompras por gênero:\n{shopping_by_genre.to_string(index=False)}')
gender_by_marital_status = (
    f.groupby([
        'Gênero',
        'Estado Civil'
    ])
    .size()
    .reset_index(name='Quantidade')
)
print(f'\nCompras agrupadas pos gênero e estado civil:\n{gender_by_marital_status.to_string(index=False)}')


Compras por gênero:
   Gênero  Quantidade
 Feminino      432576
Masculino      397424

Compras agrupadas pos gênero e estado civil:
   Gênero            Estado Civil  Quantidade
 Feminino Casado ou União Estável       90246
 Feminino              Divorciado       99896
 Feminino                Separado      113743
 Feminino                Solteiro      113020
 Feminino                   Viúvo       15671
Masculino Casado ou União Estável      104627
Masculino              Divorciado       95094
Masculino                Separado       99999
Masculino                Solteiro       89598
Masculino                   Viúvo        8106


### 4. ANÁLISE REFERENTE A QUANTIDADE ITENS POR COMPRAS.

In [37]:
average_items_per_purchase = (
    f.groupby('CO_ID')['PR_ID']
    .count()
    .mean()
)
print(f'\nMédia de itens por compra: {average_items_per_purchase:.2f}')


Média de itens por compra: 44.94


In [ ]:

products_by_marital_status = (
    products_df
    .groupby(['Estado Civil', 'Produto'])
    .size()
    .reset_index(name='Quantidade')
)

top_five_products_by_marital_status = (
    products_by_marital_status
    .sort_values(
        ['Estado Civil', 'Quantidade'],
        ascending=[True, False]
    )
    .groupby('Estado Civil')
    .head(5)
)
print(f'\nCinco produtos mais comprados separados por gênero:\n{top_five_products_by_marital_status.to_string(index=False)}')


Cinco produtos mais comprados separados por gênero:
           Estado Civil          Produto  Quantidade
Casado ou União Estável  Presunto Cozido        3333
Casado ou União Estável     Preservativo        1817
Casado ou União Estável           Cebola        1807
Casado ou União Estável         Sardinha        1786
Casado ou União Estável  Papel Higienico        1779
             Divorciado  Presunto Cozido        3362
             Divorciado         Sardinha        1797
             Divorciado           Cebola        1791
             Divorciado        Removedor        1784
             Divorciado         Linguica        1779
               Separado  Presunto Cozido        3783
               Separado        Modelador        1968
               Separado            Leite        1952
               Separado          Chupeta        1938
               Separado            Talco        1935
               Solteiro  Presunto Cozido        3506
               Solteiro       Detergente      

In [42]:
items_per_purchase_by_class = (
    f.groupby(['Classe', 'CO_ID'])['PR_ID']
    .count()
    .reset_index(name='Quantidade')
)

average_items_by_class = (
    items_per_purchase_by_class
    .groupby('Classe')['Quantidade']
    .mean()
    .round(2)
    .reset_index(name='Média de Itens')
)
print(f'\nMédia de itens X Classe\n{average_items_by_class.to_string(index=False)}')


Média de itens X Classe
Classe  Média de Itens
     A           45.40
     B           44.77
     C           45.19


### 5. sALVANDO NOVA BASE DE DADOS "LIMPA"

In [46]:
f.to_csv('../data/processed/Base Limpa.csv',sep=';')
print('\n')
print('=' * 50)
print('Novo arquivo \'Base Limpa.csv\' salvo com sucesos.')
print('=' * 50)



Novo arquivo 'Base Limpa.csv' salvo com sucesos.
